In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_id = "Qwen/Qwen3-4B-Instruct-2507"
adapter_dir = "/data/ephemeral/pro-nlp-finalproject-nlp-13/outputs/sft/qwen3_4b_v1"

tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

def chat_generate(user_text: str, system_text: str = "너는 사용자의 친구처럼 편하게 대화하는 AI야."):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text

print(chat_generate("오늘 너무 피곤한데, 기분 좀 나아지게 한마디 해줘."))

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

system
너는 사용자의 친구처럼 편하게 대화하는 AI야.
user
오늘 너무 피곤한데, 기분 좀 나아지게 한마디 해줘.
assistant
어떤 일이 있었던 건가?


In [9]:
messages = [
  {"role":"system","content":"너는 사용자의 친구처럼 편하게 대화하는 AI야."},
  {"role":"user","content":"나 오늘 발표 망한 거 같아…"},
  {"role":"assistant","content":"헉 왜? 무슨 일이 있었는데?"},
  {"role":"user","content":"질문에서 말이 막혀서 너무 창피했어."},
]


prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

out = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.7, top_p=0.9)
reply = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(reply)

그런 거면 나도 도움을 줄 수 있을까?


In [17]:
messages = [
  {"role":"system","content":"너는 사용자의 친구처럼 편하게 대화하는 AI야. 맥락을 읽고, 적절한 답을 해줘. 단, 여행 관련 질문만 할 예정이야"},
  {"role":"user","content":"나 이번주에 부산갈라고…"},
  {"role":"assistant","content":"오 좋은데??"},
  {"role":"user","content":"너 저번에 부산 갔다왔다며. 어디어디 갈지 나 추천좀 해줘"},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

out = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.7, top_p=0.9)
reply = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(reply)

그럼 부산역에서 버스 타고 가는 게 어떰?
